In [1]:
import os
from google.colab import drive
import zipfile

drive.mount('/content/drive')

ZIP_PATH = './data/Anomaly_Validation_Datasets.zip'
EXTRACT_DIR = './data/datasets'

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR)
    print("Unzipping datasets... this might take a minute.")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("Unzip complete!")
else:
    print("Datasets already extracted.")

Mounted at /content/drive
Unzipping datasets... this might take a minute.
Unzip complete!


In [2]:
import sys, os
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import auc, roc_curve, precision_recall_curve

SHARED_FOLDER_NAME = 'Image Segmentation Project/Project/Task 7'
EVAL_FOLDER_PATH = './eval'
sys.path.append(EVAL_FOLDER_PATH)

from erfnet import ERFNet
from transform import ToLabel

def calculate_anomaly_metrics(y_true, y_scores, ignore_index=255):
    y_true_flat = y_true.flatten()
    y_scores_flat = y_scores.flatten()

    valid_mask = (y_true_flat != ignore_index)
    y_true_filtered = y_true_flat[valid_mask]
    y_scores_filtered = y_scores_flat[valid_mask]

    # Force binarization: 0 stays 0, anything else (like 2) becomes 1
    y_true_filtered = (y_true_filtered > 0).astype(int)

    if len(y_true_filtered) == 0 or len(np.unique(y_true_filtered)) < 2:
        return 0.0, 0.0

    precision, recall, _ = precision_recall_curve(y_true_filtered, y_scores_filtered)
    auprc = auc(recall, precision)

    fpr, tpr, thresholds = roc_curve(y_true_filtered, y_scores_filtered)
    idx = np.where(tpr >= 0.95)[0][0]

    return auprc, fpr[idx]

def get_msp_score(logits):
    probs = F.softmax(logits, dim=1)
    max_probs, _ = torch.max(probs, dim=1)
    return (1.0 - max_probs).cpu().numpy()

def get_maxlogit_score(logits):
    max_logits, _ = torch.max(logits, dim=1)
    return (-max_logits).cpu().numpy()

def get_max_entropy_score(logits):
    probs = F.softmax(logits, dim=1)

    # Calculate raw entropy (with 1e-8 for numerical stability to prevent log(0) crashes)
    raw_entropy = torch.sum(-probs * torch.log(probs + 1e-8), dim=1)

    # Divide by log(C) to normalize to [0, 1]
    num_classes = probs.shape[1]
    normalized_entropy = torch.div(raw_entropy, torch.log(torch.tensor(num_classes, dtype=torch.float32, device=logits.device)))

    return normalized_entropy.cpu().numpy()

class CustomAnomalyDataset(Dataset):
    def __init__(self, root, input_transform=None, target_transform=None):
        self.images_dir = os.path.join(root, 'images')
        self.masks_dir = os.path.join(root, 'labels_masks')
        image_files = sorted([f for f in os.listdir(self.images_dir) if not f.startswith('.')])
        self.samples = []
        for img_file in image_files:
            stem = os.path.splitext(img_file)[0]
            mask_files = [m for m in os.listdir(self.masks_dir) if m.startswith(stem + '.')]
            if mask_files:
                self.samples.append((img_file, mask_files[0]))
        self.input_transform = input_transform
        self.target_transform = target_transform

    def __getitem__(self, index):
        img_name, mask_name = self.samples[index]
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, mask_name)
        with open(img_path, 'rb') as f: image = Image.open(f).convert('RGB')
        with open(mask_path, 'rb') as f: mask = Image.open(f).convert('P')
        if self.input_transform: image = self.input_transform(image)
        if self.target_transform: mask = self.target_transform(mask)
        return image, mask, img_name, mask_name

    def __len__(self):
        return len(self.samples)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 1

print("Loading ERFNet...")
model = ERFNet(20).to(device)

weights_path = './checkpoints/erfnet_pretrained.pth'
checkpoint = torch.load(weights_path, map_location=device)

cleaned_state_dict = {k.replace('module.', ''): v for k, v in checkpoint.items()}
model.load_state_dict(cleaned_state_dict, strict=False)
model.eval()
print("Weights loaded successfully!")

input_transform_fn = transforms.Compose([transforms.ToTensor()])
target_transform_fn = ToLabel()

Loading ERFNet...
Weights loaded successfully!


In [4]:
def evaluate_single_dataset(dataset_name, base_dir='./data/datasets/Validation_Dataset'):
    print(f"\n--- Evaluating on {dataset_name} ---")

    val_dataset = CustomAnomalyDataset(
        root=f'{base_dir}/{dataset_name}',
        input_transform=input_transform_fn,
        target_transform=target_transform_fn
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    all_gt, all_msp, all_maxlogit, all_entropy = [], [], [], []

    for images, masks, _, _ in tqdm(val_loader, desc=f"Processing"):
        images = images.to(device)
        logits = model(images)
        masks = masks.squeeze(1).numpy()

        # Memory optimization downcasting
        all_gt.append(masks.astype(np.uint8))
        all_msp.append(get_msp_score(logits).astype(np.float16))
        all_maxlogit.append(get_maxlogit_score(logits).astype(np.float16))
        all_entropy.append(get_max_entropy_score(logits).astype(np.float16))

        del images, logits
        torch.cuda.empty_cache()

    print(f"\nCalculating metrics for {dataset_name}...")

    # Calculate and clear memory
    gt_concat = np.concatenate(all_gt)
    del all_gt

    msp_concat = np.concatenate(all_msp)
    del all_msp
    msp_auprc, msp_fpr95 = calculate_anomaly_metrics(gt_concat, msp_concat)
    del msp_concat

    maxlogit_concat = np.concatenate(all_maxlogit)
    del all_maxlogit
    ml_auprc, ml_fpr95 = calculate_anomaly_metrics(gt_concat, maxlogit_concat)
    del maxlogit_concat

    entropy_concat = np.concatenate(all_entropy)
    del all_entropy
    ent_auprc, ent_fpr95 = calculate_anomaly_metrics(gt_concat, entropy_concat)
    del entropy_concat
    del gt_concat

    return {
        "MSP": {"AuPRC": msp_auprc, "FPR95": msp_fpr95},
        "MaxLogit": {"AuPRC": ml_auprc, "FPR95": ml_fpr95},
        "Max Entropy": {"AuPRC": ent_auprc, "FPR95": ent_fpr95}
    }

In [5]:
import pandas as pd

if 'results_table' not in locals():
    results_table = {}

datasets_to_run = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]

with torch.no_grad():
    for d_name in datasets_to_run:
        results_table[d_name] = evaluate_single_dataset(d_name)

df_results = pd.DataFrame(results_table).round(4)

print("\nRisultati Finali (AuPRC / FPR95):")
print(df_results.to_string(float_format='{:.3f}'.format))


--- Evaluating on RoadAnomaly21 ---


Processing: 100%|██████████| 10/10 [00:02<00:00,  3.95it/s]



Calculating metrics for RoadAnomaly21...

--- Evaluating on RoadObsticle21 ---


Processing: 100%|██████████| 30/30 [00:09<00:00,  3.18it/s]



Calculating metrics for RoadObsticle21...

--- Evaluating on FS_LostFound_full ---


...


Calculating metrics for FS_LostFound_full...

--- Evaluating on fs_static ---


Processing: 100%|██████████| 30/30 [00:07<00:00,  4.23it/s]



Calculating metrics for fs_static...

--- Evaluating on RoadAnomaly ---


Processing: 100%|██████████| 60/60 [00:06<00:00,  8.71it/s]



Calculating metrics for RoadAnomaly...

Risultati Finali (AuPRC / FPR95):
                                                           RoadAnomaly21                                                 RoadObsticle21                                             FS_LostFound_full                                                     fs_static                                                  RoadAnomaly
MSP          {'AuPRC': 0.23966489331393975, 'FPR95': 0.7230624581850428}  {'AuPRC': 0.0066692793655520576, 'FPR95': 0.8971199356019548}    {'AuPRC': 0.01296874074062327, 'FPR95': 0.798897574067211}  {'AuPRC': 0.034853012433016174, 'FPR95': 0.6332891060420006}  {'AuPRC': 0.10773456230481289, 'FPR95': 0.8847792058939224}
MaxLogit       {'AuPRC': 0.318967273132853, 'FPR95': 0.7243994811316398}   {'AuPRC': 0.011780005954063422, 'FPR95': 0.8030758650216134}  {'AuPRC': 0.025541032157706877, 'FPR95': 0.7216582463386632}   {'AuPRC': 0.04235570869450227, 'FPR95': 0.6588934617536364}  {'AuPRC': 0.1330869882

In [6]:
from torchvision.datasets import Cityscapes
from torchvision.transforms import Compose, Resize, ToTensor
from PIL import Image
import torch, time

import sys
sys.path.insert(0, '/content/drive/MyDrive/Image Segmentation Project/Project/MaskArchitectureAnomaly_CourseProject/eval')
from iouEval import iouEval

CS_ROOT = "/content/drive/MyDrive/Image Segmentation Project/Project/2. Cityscapes"
NUM_CLASSES = 20
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

input_transform = Compose([
    Resize((512, 1024), Image.BILINEAR),
    ToTensor(),
])

cs_val = Cityscapes(CS_ROOT, split='val', mode='fine',
                    target_type='semantic',
                    transform=input_transform)

import numpy as np

def collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    labels = torch.stack([
        torch.from_numpy(np.array(
            item[1].resize((1024, 512), Image.NEAREST),
            dtype=np.int64
        ))
        for item in batch
    ])
    return images, labels

loader = torch.utils.data.DataLoader(cs_val, batch_size=1,
                                      shuffle=False, num_workers=2,
                                      collate_fn=collate_fn)

iou_eval = iouEval(NUM_CLASSES)
start = time.time()

with torch.no_grad():
    for step, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels_train = torch.full_like(labels, 19, dtype=torch.long)
        for cls in Cityscapes.classes:
            if not cls.ignore_in_eval:
                labels_train[labels == cls.id] = cls.train_id
        labels_train = labels_train.to(device)

        outputs = model(images)
        iou_eval.addBatch(outputs.max(1)[1].unsqueeze(1).data,
                          labels_train.unsqueeze(1))

        if step % 100 == 0:
            print(f"[{step}/500] {time.time()-start:.0f}s")

iou_val, _ = iou_eval.getIoU()
print(f"\nERFNet mIoU: {iou_val*100:.1f}%")

[0/500] 5s
[100/500] 46s
[200/500] 77s
[300/500] 118s
[400/500] 171s

ERFNet mIoU: 72.2%
